# Day 2 — Python for Network & System Security

*Python for Security — 3-Day Intensive  |  Istidama Consulting*

**DELTA Stage:** Learn + Test

### Day Overview
Day 2 moves from text files to live systems: HTTP requests, structured network logs, and read-only OS interaction. The day closes with an ethics briefing and a hands-on build lab where each learner ships one small, complete security tool of their choosing.

Each day in this notebook is organized in two parts: **Techniques** (worked, runnable examples you read and run) followed by **Practice — Your Turn** (the floor/ceiling exercises you complete yourself, gathered at the end).

---


## 2.1 Recap & Repo Check  *(15 min)*

**Objective:** Surface and correct common Day 1 mistakes before building on them.

**Steps:**

1. Quick show of hands on who finished each Day 1 deliverable.
2. Spot-check 2–3 repos and address the most common commit-hygiene or regex issue as a group.


## 2.2 Techniques Demonstrated: HTTP Requests, CSV Parsing & OS Interaction  *(45 min — read along and run every cell)*

**Objective:** See each core technique work end to end before applying it yourself in Practice.

**What's demonstrated below, in order:**

1. Making an HTTP request with `requests.get()` and inspecting the response.
2. Looping over multiple URLs and flagging non-200 responses.
3. Reading a CSV with `csv.DictReader` and tallying a column with `Counter`.
4. Flagging rows that match a watchlist of values.
5. Listing running processes (with a safe fallback if `psutil` isn't installed).
6. Running a read-only command with `subprocess.run()`.
7. Hashing a file with `hashlib.sha256()` and comparing digests.
8. Checking whether a local port is open with `socket`.

These are the exact techniques you'll apply yourself — against different data and targets — in the Practice section at the end of this notebook.


In [1]:
import requests  # third-party HTTP library

# Technique 1: make a request, inspect status_code / headers / text
demo_url = "https://httpbin.org/json"  # a public test endpoint returning JSON
response = requests.get(demo_url)  # issue an HTTP GET request

print("status code:", response.status_code)  # the numeric HTTP status returned
print("headers (subset):", dict(list(response.headers.items())[:3]))  # first 3 response headers as a dict
print("body preview:", response.text[:100])  # first 100 characters of the response body


status code: 200
headers (subset): {'Date': 'Thu, 10 Sep 2026 06:43:36 GMT', 'Content-Type': 'application/json', 'Content-Length': '429'}
body preview: {
  "slideshow": {
    "author": "Yours Truly", 
    "date": "date of publication", 
    "slides": [


In [2]:
# Technique 2: loop over multiple URLs, flag anything that isn't 200
demo_urls = [
    "https://httpbin.org/status/200",  # endpoint that always returns 200
    "https://httpbin.org/status/403",  # endpoint that always returns 403
    "https://httpbin.org/status/404",  # endpoint that always returns 404
]  # list of test endpoints to check

for u in demo_urls:  # loop over each URL
    r = requests.get(u)  # request the URL
    flag = "" if r.status_code == 200 else "  <-- FLAGGED"  # mark anything that isn't a 200 OK
    print(f"{u} -> {r.status_code}{flag}")  # print the URL, its status, and any flag


https://httpbin.org/status/200 -> 200
https://httpbin.org/status/403 -> 403  <-- FLAGGED
https://httpbin.org/status/404 -> 404  <-- FLAGGED


In [3]:
import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Demo setup: a small connections CSV, separate from the one you'll practice on later
demo_csv = """timestamp,src_ip,dst_port,protocol,bytes
2026-02-10T10:00:01,192.168.1.5,443,tcp,500
2026-02-10T10:00:04,192.168.1.6,53,udp,120
2026-02-10T10:00:09,192.168.1.5,443,tcp,600
2026-02-10T10:00:15,192.168.1.7,8080,tcp,300
2026-02-10T10:00:22,192.168.1.5,53,udp,90
"""  # multi-line CSV string: header row plus five sample connection rows

with open("demo_connections.csv", "w") as f:  # open (create/overwrite) the demo CSV for writing
    f.write(demo_csv)  # write the sample CSV text to disk

# Technique 3: csv.DictReader + Counter tally
with open("demo_connections.csv") as f:  # open the CSV for reading
    reader = csv.DictReader(f)  # parse rows into dicts keyed by the header column names
    protocol_counts = Counter(row["protocol"] for row in reader)  # tally how many rows use each protocol

print(protocol_counts)  # print the full Counter mapping
print("Most common:", protocol_counts.most_common(2))  # print the two most common protocols


Counter({'tcp': 3, 'udp': 2})
Most common: [('tcp', 3), ('udp', 2)]


In [4]:
# Technique 4: flag rows whose value matches a watchlist set
watchlist_ports = {"53", "8080"}  # set of destination ports to watch for

with open("demo_connections.csv") as f:  # open the CSV for reading
    reader = csv.DictReader(f)  # parse rows into dicts keyed by the header column names
    for row in reader:  # iterate over each connection row
        if row["dst_port"] in watchlist_ports:  # check if this row's port is on the watchlist
            print("watchlist hit:", row)  # print the full matching row


watchlist hit: {'timestamp': '2026-02-10T10:00:04', 'src_ip': '192.168.1.6', 'dst_port': '53', 'protocol': 'udp', 'bytes': '120'}
watchlist hit: {'timestamp': '2026-02-10T10:00:15', 'src_ip': '192.168.1.7', 'dst_port': '8080', 'protocol': 'tcp', 'bytes': '300'}
watchlist hit: {'timestamp': '2026-02-10T10:00:22', 'src_ip': '192.168.1.5', 'dst_port': '53', 'protocol': 'udp', 'bytes': '90'}


In [5]:
# Technique 5: list running processes, with a safe fallback
try:
    import psutil  # optional third-party library for cross-platform process listing
    procs = [p.info for p in psutil.process_iter(["pid", "name"])][:5]  # grab pid/name for the first 5 processes
    for p in procs:  # loop over the collected process info dicts
        print(p)  # print each process's info
except ImportError:  # psutil isn't installed -- fall back to a shell command
    import subprocess  # standard library module for running external commands
    result = subprocess.run(["ps", "aux"], capture_output=True, text=True)  # run `ps aux` and capture its output
    for line in result.stdout.splitlines()[:5]:  # take the first 5 lines of output
        print(line)  # print each line


{'name': 'System Idle Process', 'pid': 0}
{'name': 'System', 'pid': 4}
{'name': '', 'pid': 188}
{'name': 'Registry', 'pid': 228}
{'name': 'smss.exe', 'pid': 820}


In [7]:
import subprocess
import sys  # standard library module for running external commands

# Technique 6: run a safe, read-only command and capture its output
#result = subprocess.run(["echo", "read-only demo command"], capture_output=True, text=True) this runs on linux # run echo and capture output
result = subprocess.run([sys.executable, "-c", "print('read-only demo command')"], capture_output=True, text=True)
print("return code:", result.returncode)  # 0 means the command succeeded
print("stdout:", result.stdout.strip())  # the command's captured output, trailing whitespace removed


return code: 0
stdout: read-only demo command


In [8]:
import hashlib  # standard library module providing cryptographic hash functions

# Technique 7: hash a file, then verify it against a known digest
with open("demo_note.txt", "w") as f:  # open (create/overwrite) a demo text file
    f.write("Techniques demo file -- do not use for the hashing lab.\n")  # write a line of sample content

with open("demo_note.txt", "rb") as f:  # reopen the file in binary mode for hashing
    digest = hashlib.sha256(f.read()).hexdigest()  # compute the file's SHA-256 hash as a hex string
print("sha256:", digest)  # print the computed hash

known_good = digest  # pretend this hash was recorded earlier as the "known good" value
with open("demo_note.txt", "rb") as f:  # reopen the file again in binary mode
    recomputed = hashlib.sha256(f.read()).hexdigest()  # recompute the hash to check for changes
print("match?", recomputed == known_good)  # True if the file is unchanged since known_good was recorded


sha256: ed5a3b078c106a5349b2d9eaa68bd757f30acf2d808a48110d05461647abe313
match? True


In [ ]:
import socket  # standard library module for low-level networking

# Technique 8: check whether a port is open, LOCAL TARGET ONLY
def port_status(host, port, timeout=0.5):  # define a helper to check a single host:port
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)  # create a TCP/IPv4 socket
    sock.settimeout(timeout)  # cap how long to wait for a connection attempt (if it doent finish within this time dont wait it)
    result = sock.connect_ex((host, port))  # try to connect; returns 0 on success, an error code otherwise
    sock.close()  # release the socket
    return "open" if result == 0 else "closed"  # translate the result code into a human-readable status

for port in [22, 9999]:  # check a couple of sample ports
    print(f"127.0.0.1:{port} -> {port_status('127.0.0.1', port)}")  # print the status of each port


127.0.0.1:22 -> closed
127.0.0.1:9999 -> closed


## Examples

Work through the practice items below using the techniques demonstrated above. Floor/ceiling markers tell you the minimum bar and the stretch goal.


### 2.3 Practice: Networking with Python  *(30 min)*

**Objective:** Make and interpret basic HTTP requests in Python.

**Steps:**

1. Practice: hit 2–3 public test endpoints and print their status codes.
2. Discuss what a 200 vs. 403 vs. 404 vs. 500 response actually tells you.

> **Floor:** script prints the status code and first 100 characters of the response body for a given URL.
> **Ceiling:** script checks a list of URLs and flags any that don't return 200.


In [ ]:
import requests  # third-party HTTP library

# Floor: fetch one URL and inspect the response
url = "https://httpbin.org/status/200"  # a public test endpoint that always returns 200
response = requests.get(url)  # issue an HTTP GET request

print("status code:", response.status_code)  # the numeric HTTP status returned
print("body preview:", response.text[:100])  # first 100 characters of the response body

the output is status code: 200
body preview: here the developer didnt put a body but the code works sucssfully

status code: 200
body preview: 


In [11]:
# Ceiling: check a list of URLs and flag any that don't return 200
urls = [
    "https://httpbin.org/status/200",  # endpoint that always returns 200
    "https://httpbin.org/status/403",  # endpoint that always returns 403
    "https://httpbin.org/status/404",  # endpoint that always returns 404
    "https://httpbin.org/status/500",  # endpoint that always returns 500
]  # list of test endpoints to check
for u in urls:  # loop over each URL
    r = requests.get(u)  # request the URL
    flag = "" if r.status_code == 200 else "  <-- FLAGGED"  # mark anything that isn't a 200 OK
    print(f"{u} -> {r.status_code}{flag}")  # print the URL, its status, and any flag


https://httpbin.org/status/200 -> 200
https://httpbin.org/status/403 -> 403  <-- FLAGGED
https://httpbin.org/status/404 -> 404  <-- FLAGGED
https://httpbin.org/status/500 -> 500  <-- FLAGGED


### 2.4 Practice: Parsing Network/Traffic Log Data  *(45 min)*

**Objective:** Turn a raw structured log into a short summary report.

**Materials:** sample_traffic.csv (generated below)

**Steps:**

1. Run the setup cell to generate sample_traffic.csv.
2. Read rows using the csv module.
3. Count connections per destination port.
4. Print a short report: top 5 ports by connection count.

> **Floor:** report prints correctly for the sample file.
> **Ceiling:** report also flags traffic to a short instructor-provided list of “suspicious” ports (e.g., 4444, 31337).


In [13]:
# Setup: generate a sample traffic CSV
traffic_csv = """timestamp,src_ip,dst_port,protocol,bytes
2026-01-04T09:00:01,10.0.0.5,443,tcp,1200
2026-01-04T09:00:05,10.0.0.6,80,tcp,850
2026-01-04T09:00:09,10.0.0.5,443,tcp,900
2026-01-04T09:00:12,10.0.0.9,4444,tcp,50
2026-01-04T09:00:14,10.0.0.5,22,tcp,300
2026-01-04T09:00:20,10.0.0.9,4444,tcp,60
2026-01-04T09:00:25,10.0.0.7,80,tcp,700
"""  # multi-line CSV string: header row plus seven sample traffic rows (includes port 4444 twice)

with open("sample_traffic.csv", "w") as f:  # open (create/overwrite) the traffic CSV for writing
    f.write(traffic_csv)  # write the sample CSV text to disk


In [14]:
import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Floor: top 5 destination ports by connection count
with open("sample_traffic.csv") as f:  # open the traffic CSV for reading
    reader = csv.DictReader(f)  # parse rows into dicts keyed by the header column names
    port_counts = Counter(row["dst_port"] for row in reader)  # tally how many connections hit each port

print("Top 5 ports:", port_counts.most_common(5))  # print the five most-hit destination ports


Top 5 ports: [('443', 2), ('80', 2), ('4444', 2), ('22', 1)]


In [ ]:
# Ceiling: flag traffic to suspicious ports (we wanna open a file and see if there is any suspicious port in the traffic and detect it)
suspicious_ports = {"4444", "31337"}  # set of ports commonly associated with malware C2
with open("sample_traffic.csv") as f:  # open the traffic CSV for reading
    reader = csv.DictReader(f)  # parse rows into dicts keyed by the header column names
    for row in reader:  # iterate over each traffic row
        if row["dst_port"] in suspicious_ports:  # check if this row's port is suspicious
            print("SUSPICIOUS:", row)  # print the full matching row


SUSPICIOUS: {'timestamp': '2026-01-04T09:00:12', 'src_ip': '10.0.0.9', 'dst_port': '4444', 'protocol': 'tcp', 'bytes': '50'}
SUSPICIOUS: {'timestamp': '2026-01-04T09:00:20', 'src_ip': '10.0.0.9', 'dst_port': '4444', 'protocol': 'tcp', 'bytes': '60'}


### 2.5 Practice: OS Interaction — Processes & Hashing  *(45 min)*

**Objective:** Read OS-level information and verify file integrity from Python.

**Steps:**

1. List a few running processes read-only (os or psutil, whichever is pre-installed).
2. Use subprocess.run() to call one safe, read-only command and capture its output.
3. Use hashlib.sha256() to hash a sample file and print the digest.

> **Floor:** process list printed and a file hash computed.
> **Ceiling:** script compares a file's hash against a known-good value and reports match/mismatch.

*Discussion prompt: Why hash a file before and after a suspected incident? What does a mismatch tell an investigator?*


In [ ]:
import subprocess  # standard library module for running external commands
#this is ex of python talking to os
# Floor part 1: list running processes (read-only)
try:
    import psutil  # optional third-party library for cross-platform process listing
    for p in psutil.process_iter(["pid", "name"]):  # iterate over all running processes
        print(p.info)  # print each process's pid and name
except ImportError:  # psutil isn't installed -- fall back to a shell command
    result = subprocess.run(["ps", "aux"], capture_output=True, text=True)  # run `ps aux` and capture its output
    for line in result.stdout.splitlines()[:10]:  # take the first 10 lines of output
        print(line)  # print each line


{'name': 'System Idle Process', 'pid': 0}
{'name': 'System', 'pid': 4}
{'name': '', 'pid': 188}
{'name': 'Registry', 'pid': 228}
{'name': 'smss.exe', 'pid': 820}
{'name': 'svchost.exe', 'pid': 908}
{'name': 'chrome.exe', 'pid': 992}
{'name': 'svchost.exe', 'pid': 1080}
{'name': 'csrss.exe', 'pid': 1096}
{'name': 'svchost.exe', 'pid': 1176}
{'name': 'svchost.exe', 'pid': 1180}
{'name': 'wininit.exe', 'pid': 1184}
{'name': 'services.exe', 'pid': 1304}
{'name': 'svchost.exe', 'pid': 1336}
{'name': 'LsaIso.exe', 'pid': 1348}
{'name': 'lsass.exe', 'pid': 1356}
{'name': 'OpenConsole.exe', 'pid': 1412}
{'name': 'svchost.exe', 'pid': 1504}
{'name': 'chrome.exe', 'pid': 1532}
{'name': 'fontdrvhost.exe', 'pid': 1564}
{'name': 'UserOOBEBroker.exe', 'pid': 1644}
{'name': 'svchost.exe', 'pid': 1668}
{'name': 'svchost.exe', 'pid': 1724}
{'name': 'NisSrv.exe', 'pid': 1760}
{'name': 'WUDFHost.exe', 'pid': 1848}
{'name': 'Code.exe', 'pid': 1924}
{'name': 'svchost.exe', 'pid': 2020}
{'name': 'svchost.ex

In [ ]:
import hashlib  # standard library module providing cryptographic hash functions
#how to hash a file and verify it against a known digest
# Setup: a sample file to hash
with open("sample_file.txt", "w") as f:  # open (create/overwrite) the sample file for writing
    f.write("This is a sample file for hashing practice.\n")  # write a line of sample content

# Floor part 2: hash sample_file.txt with sha256
with open("sample_file.txt", "rb") as f:  # reopen the file in binary mode for hashing
    digest = hashlib.sha256(f.read()).hexdigest()  # compute the file's SHA-256 hash as a hex string
print("sha256:", digest)  # print the computed hash


sha256: 07a2b73c916791807f45ca6aa40cd1b8ed96b2439e31a47f793e0bdc45976f01


In [ ]:
# Ceiling: compare against a known-good hash
known_good_hash = digest  # reuse the hash computed in the previous cell as the "known good" value
# here we simulate a file change by overwriting the file with new content to check compare files if it changed or not
with open("sample_file.txt", "rb") as f:  # reopen the file in binary mode for hashing
    recomputed = hashlib.sha256(f.read()).hexdigest()  # recompute the hash to check for changes

print("recomputed:", recomputed)  # print the freshly computed hash
print("match?", recomputed == known_good_hash)  # True if the file is unchanged since known_good_hash was recorded


recomputed: 07a2b73c916791807f45ca6aa40cd1b8ed96b2439e31a47f793e0bdc45976f01
match? True


## 2.6 Ethics & Authorized-Use Briefing  *(15 min — required before 2.7)*

**Objective:** Set explicit boundaries before learners build anything that probes or scans.

**Steps:**

1. State the rule plainly: these tools are only ever run against systems you own or are explicitly authorized to test.
2. Confirm the classroom exercises target only instructor-provided sample data and local test targets — never live third-party systems.
3. Be ready to name a real scenario where running this without authorization would be illegal or unethical.

*Instructor note: Do not move on to 2.7 until this briefing has landed with the whole room, including the remote stream.*


### 2.7 Practice: Build Lab — Choose Your Tool  *(90 min)*

**Objective:** Build one small, complete security tool end to end.

**Steps:**

1. Peer review: pair up, trade scripts, and leave two comments each — one compliment, one suggestion.

**Learner picks ONE:**

- Hash Verifier — floor: verify a single file's hash against a provided value; ceiling: verify a whole directory and report any mismatches.
- Log Anomaly Flagger — floor: flag lines matching a keyword list; ceiling: flag any IP exceeding a failed-attempt threshold.
- Basic Port Scanner (local test target only) — floor: check a small fixed list of ports against a provided local test address; ceiling: scan a small range and report open/closed cleanly.


In [ ]:
import hashlib  # standard library module providing cryptographic hash functions
import os  # standard library module for filesystem paths and checks

# Option A: Hash Verifier
def verify_file(path, expected_hash):  # define a function comparing one file's hash to an expected value
    """Return True if path's sha256 matches expected_hash."""
    with open(path, "rb") as f:  # open the file in binary mode for hashing
        actual_hash = hashlib.sha256(f.read()).hexdigest()  # compute the file's actual SHA-256 hash
    return actual_hash == expected_hash  # True only if the hashes match exactly

def verify_directory(dir_path, expected_hashes: dict):  # define a function checking a whole directory
    """expected_hashes maps filename -> expected sha256. Print mismatches. (ceiling)"""
    for filename, expected_hash in expected_hashes.items():  # loop over each expected filename/hash pair
        path = os.path.join(dir_path, filename)  # build the full path to the file
        if not os.path.isfile(path):  # the expected file doesn't exist at all
            print(f"MISSING: {filename}")  # report it as missing
        elif not verify_file(path, expected_hash):  # the file exists but its hash doesn't match
            print(f"MISMATCH: {filename}")  # report the mismatch
        else:  # the file exists and its hash matches
            print(f"OK: {filename}")  # report it as verified


In [19]:
import re  # regular expression module for pattern matching
from collections import Counter  # Counter tallies hashable items

# Option B: Log Anomaly Flagger
def flag_keywords(filename, keywords):  # define a function flagging lines matching any keyword
    """Print any line in filename containing one of keywords. (floor)"""
    with open(filename) as f:  # open the target file for reading
        for line in f:  # iterate line by line
            if any(keyword in line for keyword in keywords):  # true if any keyword appears in this line
                print(line.strip())  # print the line without its trailing newline

def flag_ip_threshold(filename, threshold):  # define a function flagging high-volume offending IPs
    """Flag any source IP with more than `threshold` failed attempts. (ceiling)"""
    ip_counts = Counter()  # empty tally of failed attempts per IP
    with open(filename) as f:  # open the target file for reading
        for line in f:  # iterate line by line
            if "Failed password" in line:  # only consider failed-login lines
                match = re.search(r"\d+\.\d+\.\d+\.\d+", line)  # try to find an IP address in the line
                if match:  # only tally if an IP was actually found
                    ip_counts[match.group()] += 1  # increment that IP's failed-attempt count
    for ip, count in ip_counts.items():  # loop over every IP and its tally
        if count > threshold:  # only report IPs that exceed the threshold
            print(f"FLAGGED: {ip} ({count} failed attempts)")  # report the offending IP and its count


In [20]:
import socket  # standard library module for low-level networking

# Option C: Basic Port Scanner (LOCAL TEST TARGET ONLY -- e.g. "127.0.0.1")
def check_ports(host, ports):  # define a function checking a fixed list of ports on host
    """Print open/closed for each port in ports on host. (floor)"""
    for port in ports:  # loop over each port to check
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)  # create a TCP/IPv4 socket
        sock.settimeout(0.5)  # cap how long to wait for a connection attempt
        result = sock.connect_ex((host, port))  # try to connect; returns 0 on success, an error code otherwise
        sock.close()  # release the socket
        status = "open" if result == 0 else "closed"  # translate the result code into a human-readable status
        print(f"{host}:{port} -> {status}")  # print the host:port and its status

def scan_range(host, start_port, end_port):  # define a function scanning a contiguous range of ports
    """Scan a small range and report open/closed cleanly. (ceiling)"""
    check_ports(host, range(start_port, end_port + 1))  # reuse check_ports over the inclusive port range


## 2.8 Commit, Push, Preview Day 3  *(15 min)*

**Objective:** Close the day with version control and a look ahead to DFIR work.

**Steps:**

1. Stage, commit, and push the chosen tool.
2. Read ahead: Day 3's forensics work builds directly on today's hashing and OS-interaction skills.


```bash
git add .
git commit -m "Add Day 2 security tool"
git push
```


## 2.9 Exercises — On Your Own

New problems, separate from the build lab above. Use the techniques from section 2.2 (`requests`, `csv.DictReader`, `Counter`, `hashlib`, `socket`). No worked example to copy from this time.

> Do these after the build lab, as homework, or as a fast-finisher extension.


**Exercise 1 — JSON field extraction.** `GET https://httpbin.org/uuid` returns a JSON body like `{"uuid": "..."}`. Make the request, parse the JSON with `response.json()` (instead of slicing `response.text`), and print just the `uuid` value.

In [ ]:
import requests  # third-party HTTP library

# Exercise 1: fetch https://httpbin.org/uuid and print just the "uuid" field



**Exercise 2 — Top talker by bytes.** Using `exercise_connections.csv` (generated below), find the `src_ip` with the highest *total* `bytes` transferred, not just the most connections. *Hint: `Counter` supports addition via indexing — `counts[key] += int(value)` — it isn't only for counting occurrences.*

In [ ]:
# Setup: a fresh connections CSV for this exercise
exercise_connections_csv = """timestamp,src_ip,dst_port,protocol,bytes
2026-03-02T11:00:01,10.2.0.5,443,tcp,4000
2026-03-02T11:00:04,10.2.0.6,53,udp,120
2026-03-02T11:00:09,10.2.0.5,443,tcp,6500
2026-03-02T11:00:15,10.2.0.7,8080,tcp,300
2026-03-02T11:00:22,10.2.0.6,53,udp,90
2026-03-02T11:00:30,10.2.0.5,443,tcp,2200
"""  # multi-line CSV string: header row plus six sample rows (one IP sends most of the bytes)

with open("exercise_connections.csv", "w") as f:  # open (create/overwrite) the exercise CSV for writing
    f.write(exercise_connections_csv)  # write the sample CSV text to disk

import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items, including running totals

# Exercise 2: find the src_ip with the highest total bytes transferred



**Exercise 3 — Stretch: combine hashing and a port check.** Write a function `quick_check(path, host, port)` that returns a dict with two keys: `"sha256"` (the file's hash) and `"port_status"` (`"open"` or `"closed"`, reusing the `port_status()` pattern from section 2.2, technique 8). Call it on `sample_file.txt` against `127.0.0.1` port `22`, and print the result.

In [ ]:
import hashlib  # standard library module providing cryptographic hash functions
import socket  # standard library module for low-level networking

def quick_check(path, host, port):  # define the function signature for this exercise
    """Return {'sha256': ..., 'port_status': 'open'|'closed'}."""
    # Exercise 3: implement this function, then call it below
    pass  # placeholder -- replace with your implementation

# quick_check("sample_file.txt", "127.0.0.1", 22)
